
# DBSCAN on 5 Phone Classes Excluding `sil`

This notebook follows the professor's suggestion to work with a **smaller 5-class phone subset first**.

Goal:
- exclude `sil`
- keep only **5 dense phone classes**
- visualize the **raw data** with t-SNE using **ground-truth phone labels**
- then run DBSCAN and compare its clusters against that same t-SNE geometry
- tune DBSCAN toward **5 clusters** instead of trying to solve all ~40 speech classes at once

Default choice in this notebook:
- `s`
- `ih`
- `aa`
- `iy`
- `ae`

These were chosen because they are the **top 5 non-silence phone classes by frame count** in `train_dim12.json`, so they are a sensible dense starting point.


In [ ]:

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    completeness_score,
    homogeneity_score,
    normalized_mutual_info_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

plt.style.use("ggplot")

with open("train_dim12.json", "r") as f:
    data = json.load(f)

RANDOM_SEED = 0
SELECTED_PHONEMES = ["s", "ih", "aa", "iy", "ae"]
SUBSET_SIZE = 12_000
TSNE_SIZE = 3_500
TARGET_CLUSTERS = 5

print("num utterances:", len(data))
print("selected phonemes:", SELECTED_PHONEMES)
print("subset size:", SUBSET_SIZE)



## Global Phone Counts

Before restricting to 5 classes, check which phone labels are most frequent at the **frame level**.
This justifies the default subset choice.


In [ ]:

def expand_frame_labels(utt):
    T = len(utt["features"])
    frame_labels = np.full(T, "unlabeled", dtype=object)
    transcript = utt["phn_transcript"]

    for seg_idx, (label, start, end) in enumerate(transcript):
        start = max(0, min(int(start), T))
        end = max(start, min(int(end), T))
        if seg_idx == len(transcript) - 1 and end == T - 1:
            end = T
        if end > start:
            frame_labels[start:end] = label

    return frame_labels


frame_counter = Counter()
for utt in data.values():
    frame_counter.update(expand_frame_labels(utt).tolist())

phone_count_df = pd.DataFrame(
    [(label, count) for label, count in frame_counter.items() if label not in {"sil", "unlabeled"}],
    columns=["phoneme", "frame_count"],
).sort_values("frame_count", ascending=False).reset_index(drop=True)

display(phone_count_df.head(15))
print("default 5-class subset:", phone_count_df.head(5)["phoneme"].tolist())



## Build The 5-Class Frame Dataset

We keep only frames whose expanded phone label is one of the selected 5 classes.
Each DBSCAN point is still a **raw 12D frame vector**.


In [ ]:

def build_selected_frame_dataset(dataset, selected_phonemes):
    X_parts = []
    y_parts = []
    meta_parts = []

    selected_set = set(selected_phonemes)

    for utt_id, utt in dataset.items():
        feats = np.asarray(utt["features"], dtype=np.float32)
        frame_labels = expand_frame_labels(utt)
        mask = np.isin(frame_labels, list(selected_set))

        X_parts.append(feats[mask])
        y_parts.append(frame_labels[mask])
        meta_parts.append(pd.DataFrame({
            "utt_id": utt_id,
            "frame_idx": np.arange(len(frame_labels))[mask],
            "phoneme": frame_labels[mask],
        }))

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    meta = pd.concat(meta_parts, ignore_index=True)
    return X, y, meta


X_frames, y_frames, meta_df = build_selected_frame_dataset(data, SELECTED_PHONEMES)

class_balance_df = pd.Series(y_frames).value_counts().rename_axis("phoneme").reset_index(name="frame_count")

print("frame matrix shape:", X_frames.shape)
print("num classes retained:", len(np.unique(y_frames)))
display(class_balance_df)



DBSCAN tuning on all retained frames would still be slower than necessary, so we use a fixed reproducible subset.


In [ ]:

rng = np.random.default_rng(RANDOM_SEED)
subset_size = min(SUBSET_SIZE, len(X_frames))
subset_idx = rng.choice(len(X_frames), size=subset_size, replace=False)

X_sub = X_frames[subset_idx]
y_sub = y_frames[subset_idx]
meta_sub = meta_df.iloc[subset_idx].reset_index(drop=True)

scaler = StandardScaler()
X_sub_scaled = scaler.fit_transform(X_sub)

print("subset shape:", X_sub.shape)
print("scaled shape:", X_sub_scaled.shape)
print("subset class balance:")
display(pd.Series(y_sub).value_counts().rename_axis("phoneme").reset_index(name="count"))



## t-SNE of The Raw Data With Ground-Truth Phone Labels

This is the first plot requested by the professor.

Important: DBSCAN is **not** used here.
This is just the raw 12D frame data, after standardization, embedded into 2D and colored by the true phone labels.


In [ ]:

tsne_rng = np.random.default_rng(RANDOM_SEED)
vis_size = min(TSNE_SIZE, len(X_sub_scaled))
vis_idx = tsne_rng.choice(len(X_sub_scaled), size=vis_size, replace=False)

X_vis = X_sub_scaled[vis_idx]
y_vis = y_sub[vis_idx]

raw_embedding = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_SEED,
).fit_transform(X_vis)

label_codes = pd.Categorical(y_vis, categories=SELECTED_PHONEMES).codes
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    raw_embedding[:, 0],
    raw_embedding[:, 1],
    c=label_codes,
    cmap="tab10",
    s=10,
    alpha=0.75,
)
ax.set_title("Raw 12D frames: t-SNE colored by true phone label")
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")

handles, _ = scatter.legend_elements()
ax.legend(handles, SELECTED_PHONEMES, title="phone", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()



## k-Distance Heuristic For The 5-Class Subset

This helps narrow down `eps` for DBSCAN on the selected 5 classes.
Since we expect a small number of denser clusters, it makes sense to inspect neighborhood sizes near the candidate `min_samples` values.


In [ ]:

k_values = [10, 15, 20]
rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k in zip(axes, k_values):
    nbrs = NearestNeighbors(n_neighbors=k, algorithm="ball_tree")
    nbrs.fit(X_sub_scaled)
    distances, _ = nbrs.kneighbors(X_sub_scaled)
    kth_dist = np.sort(distances[:, -1])

    ax.plot(kth_dist, color="darkgreen", linewidth=1)
    ax.set_title(f"sorted distance to {k}th neighbor")
    ax.set_xlabel("sorted sample index")
    ax.set_ylabel("distance")

    quantiles = np.quantile(kth_dist, [0.5, 0.75, 0.9, 0.95, 0.99])
    rows.append({
        "k": k,
        "q50": round(float(quantiles[0]), 4),
        "q75": round(float(quantiles[1]), 4),
        "q90": round(float(quantiles[2]), 4),
        "q95": round(float(quantiles[3]), 4),
        "q99": round(float(quantiles[4]), 4),
    })

plt.tight_layout()
plt.show()
display(pd.DataFrame(rows))



## Hyperparameter Sweep Toward 5 Clusters

Now the search problem is much smaller than the full 40-class setup.
We explicitly tune for a target of **5 DBSCAN clusters**.


In [ ]:

def dbscan_sweep(X, y, eps_grid, min_samples_grid, target_clusters=5):
    rows = []
    for min_samples in min_samples_grid:
        for eps in eps_grid:
            labels = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                algorithm="ball_tree",
                n_jobs=-1,
            ).fit_predict(X)

            clustered_mask = labels >= 0
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

            rows.append({
                "min_samples": int(min_samples),
                "eps": float(eps),
                "clusters": int(n_clusters),
                "cluster_gap": int(abs(n_clusters - target_clusters)),
                "noise_fraction": round(float((labels == -1).mean()), 4),
                "coverage": round(float(clustered_mask.mean()), 4),
                "ami_all": round(float(adjusted_mutual_info_score(y, labels)), 4),
                "nmi_all": round(float(normalized_mutual_info_score(y, labels)), 4),
                "ari_all": round(float(adjusted_rand_score(y, labels)), 4),
                "homogeneity": round(float(homogeneity_score(y, labels)), 4),
                "completeness": round(float(completeness_score(y, labels)), 4),
            })
    return pd.DataFrame(rows)


eps_grid = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6]
min_samples_grid = [6, 8, 10, 12, 15, 20]

sweep_df = dbscan_sweep(
    X_sub_scaled,
    y_sub,
    eps_grid=eps_grid,
    min_samples_grid=min_samples_grid,
    target_clusters=TARGET_CLUSTERS,
)

best_by_target = sweep_df.sort_values(["cluster_gap", "noise_fraction", "eps"]).reset_index(drop=True)
display(best_by_target.head(15))



On a dry run with this exact setup, a strong candidate was:
- `eps = 1.6`
- `min_samples = 15`
- `5` clusters found
- noise fraction around `0.4288`
- coverage around `0.5712`

That is already much more tractable than the 40-class raw-frame experiment.


In [ ]:

best_row = best_by_target.iloc[0]
selected_eps = float(best_row["eps"])
selected_min_samples = int(best_row["min_samples"])

cluster_labels = DBSCAN(
    eps=selected_eps,
    min_samples=selected_min_samples,
    algorithm="ball_tree",
    n_jobs=-1,
).fit_predict(X_sub_scaled)

clustered_mask = cluster_labels >= 0
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

print("selected eps:", selected_eps)
print("selected min_samples:", selected_min_samples)
print("clusters found:", n_clusters)
print("noise fraction:", round(float((cluster_labels == -1).mean()), 4))
print("coverage:", round(float(clustered_mask.mean()), 4))
print("AMI:", round(float(adjusted_mutual_info_score(y_sub, cluster_labels)), 4))
print("NMI:", round(float(normalized_mutual_info_score(y_sub, cluster_labels)), 4))
print("ARI:", round(float(adjusted_rand_score(y_sub, cluster_labels)), 4))



## Cluster Summary

This shows which phone dominates each DBSCAN cluster and how pure each cluster is.


In [ ]:

cluster_df = meta_sub.copy()
cluster_df["cluster"] = cluster_labels
clustered_df = cluster_df[cluster_df["cluster"] >= 0].copy()

if len(clustered_df) == 0:
    print("All points were labeled as noise.")
else:
    rows = []
    for cluster_id, group in clustered_df.groupby("cluster"):
        counts = group["phoneme"].value_counts()
        rows.append({
            "cluster": int(cluster_id),
            "size": int(len(group)),
            "majority_phoneme": counts.index[0],
            "majority_fraction": round(float(counts.iloc[0] / len(group)), 4),
            "num_unique_phonemes": int(group["phoneme"].nunique()),
        })

    summary_df = pd.DataFrame(rows).sort_values(["size", "majority_fraction"], ascending=[False, False])
    display(summary_df)



## Compare Ground Truth vs DBSCAN On The Same t-SNE Embedding

This is the second visualization the professor asked for.
The embedding is computed once from the raw standardized frames, and then we color the same 2D points in two ways:
- by the true phone labels
- by the DBSCAN cluster labels


In [ ]:

cluster_vis = cluster_labels[vis_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

scatter0 = axes[0].scatter(
    raw_embedding[:, 0],
    raw_embedding[:, 1],
    c=label_codes,
    cmap="tab10",
    s=10,
    alpha=0.75,
)
axes[0].set_title("t-SNE colored by true phone label")
axes[0].set_xlabel("dim 1")
axes[0].set_ylabel("dim 2")
handles0, _ = scatter0.legend_elements()
axes[0].legend(handles0, SELECTED_PHONEMES, title="phone", bbox_to_anchor=(1.02, 1), loc="upper left")

cluster_colors = np.where(cluster_vis == -1, -1, cluster_vis)
scatter1 = axes[1].scatter(
    raw_embedding[:, 0],
    raw_embedding[:, 1],
    c=cluster_colors,
    cmap="tab10",
    s=10,
    alpha=0.75,
)
axes[1].set_title("same t-SNE colored by DBSCAN cluster")
axes[1].set_xlabel("dim 1")
axes[1].set_ylabel("dim 2")

if np.any(cluster_vis >= 0):
    unique_clusters = sorted(set(cluster_vis))
    handles1, _ = scatter1.legend_elements()
    legend_labels = [str(c) for c in unique_clusters]
    axes[1].legend(handles1[:len(legend_labels)], legend_labels, title="cluster", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()



## Takeaway

This 5-class notebook is the right place to iterate first because it is more meaningful and more tractable than forcing DBSCAN over all phone classes at once.

What to inspect:
1. whether the raw-data t-SNE already shows visible class structure for the chosen 5 phones
2. whether DBSCAN cluster colors line up with that structure on the same embedding
3. whether another 5-phone subset gives cleaner separation than the default `['s', 'ih', 'aa', 'iy', 'ae']`

If needed, the next variation is easy: just change `SELECTED_PHONEMES` in the config cell and rerun.
